In [ ]:
!nvidia-smi -L
!pip install -q ultralytics wandb

In [ ]:
from google.colab import files
import os, shutil, zipfile

In [ ]:
"""
Upload a zip file containing the scripts folder and it's contents,
the corrected folder containing the manual and corrected image annotations
(gold_test.json, train_pool_assisted.json, train_pool_manual.json), the
review_batches folder and it's contents, and pseudo_annotations.json
"""

os.chdir("/content")
uploaded = files.upload()
zip_name = next(iter(uploaded))

shutil.rmtree("/content/cup_project", ignore_errors=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall("/content/cup_project")

# if the zip had a wrapper folder, step into it automatically
root = "/content/cup_project"
if "scripts" not in os.listdir(root):
    subs = [d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]
    if len(subs) == 1:
        root = os.path.join(root, subs[0])
os.chdir(root)

print(sorted(os.listdir(".")))
assert {"scripts", "corrected", "review_batches"} <= set(os.listdir(".")), "zip layout looks wrong"

In [ ]:
# If not logging to W&B use this instead:  os.environ["USE_WANDB"] = "0"
from google.colab import userdata
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

In [ ]:
"""
Mount google drive to save run data
"""
from google.colab import drive
drive.mount("/content/drive")

os.makedirs("/content/drive/MyDrive/cup_runs_equalsteps", exist_ok=True)
if not os.path.exists("runs_cup_detector"):
    os.symlink("/content/drive/MyDrive/cup_runs_equalsteps", "runs_cup_detector")

In [ ]:
"""
Training starts when this cell runs
--force causes training to start again from scratch
--equal-steps ignores the specified number of epochs and trans each set for
 about the same number of epochs, adapted to batch size
"""
!python scripts/04_coco_to_yolo.py
!python scripts/05_train_yolo.py --epochs 70 --equal-steps --force
!python scripts/05_train_yolo.py --only assisted_30 manual_30 mixed_30 --seeds 43 44 --epochs 70 --equal-steps --force
!python scripts/05_train_yolo.py --only assisted_all mixed_all raw_all --seeds 43 44 --epochs 70 --equal-steps --force
!python scripts/06_evaluate.py

In [ ]:
files.download("evaluation_results.csv")
files.download("evaluation_results_by_seed.csv")